# RLSF reward-path smoke

---
## 1 — Setup

In [ ]:
# 7B in bf16 is ~15 GB of weights; a T4 (16 GB) is tight, an A100/L4 is comfortable.
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os
if not os.path.isdir('Style-Aware-MT'):
    !git clone --branch feat/rlsf-implementation https://github.com/prnamhr/Style-Aware-MT.git
%cd Style-Aware-MT
!git pull
!git rev-parse --short HEAD

In [ ]:
!pip install -q "transformers==5.12.1" "accelerate==1.14.0" "peft==0.18.0" \
    "sacrebleu==2.6.0" "openai==2.41.1" "PyYAML==6.0.3"

In [ ]:
# Second interpreter for COMET. The ~2.3 GB checkpoint downloads on first handshake.
import os
if not os.path.isdir('.venv-comet'):
    !python -m venv .venv-comet
    !.venv-comet/bin/pip install -q -r requirements-comet.txt
!.venv-comet/bin/python -c "import comet; print('comet ok')"

In [ ]:
import getpass, logging, os
if not os.environ.get('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass.getpass('OPENAI_API_KEY: ')
logging.getLogger('httpx').setLevel(logging.WARNING)

---
## 2 — Pre-flight


In [ ]:
import hashlib, json, pathlib, yaml
from src.rlsf.config import load_config, reward_config
from src.rlsf.reward import load_train_template
from src.rlsf.smoke import plan

CONFIG   = 'configs/rlsf.yaml'
SEGMENTS = 20

cfg = load_config(CONFIG, require_caps=False)
G   = cfg['rlsf']['rollout']['group_size']

# -- the caps gate PPO training, not this pilot; if they are set, training was authorised
#    and this notebook is the wrong tool
assert all(cfg['rlsf']['caps'][k] is None for k in
           ('max_steps', 'max_grid_steps', 'max_judge_calls', 'max_judge_spend_usd')), \
    'training caps are declared; use the training runbook, not the smoke'

# -- the locked control: a quantized or swapped base is a different experiment
gen = cfg['generator']
assert gen['model'] == 'Qwen/Qwen2.5-7B-Instruct', gen['model']
assert gen['load_in_4bit'] is False, 'quantizing redefines the frozen base'
assert gen['adapter_path'], 'RLSF initializes from the frozen PEFT checkpoint'

# -- greedy rollouts give a group zero variance and the whole run is uninformative
assert cfg['rlsf']['rollout']['temperature'] > 0, 'greedy rollouts cannot be normalized'
assert G <= cfg['rlsf']['caps']['group_size_ceiling']

print(f"policy {gen['model']} + {gen['adapter_path']}")
print(f"rollout T={cfg['rlsf']['rollout']['temperature']} G={G}")
print('reward', reward_config(cfg))

In [ ]:
# -- the reward judge must not be either evaluation rater, or training spends a rater on
#    the one condition that most needs a rater it was not trained against
raters = {yaml.safe_load(pathlib.Path(p).read_text())['judge']['model']
          for p in ('configs/judge_eval.yaml', 'configs/judge_eval_gpt.yaml')}
assert cfg['judge']['model'] not in raters, (cfg['judge']['model'], raters)

# -- seeded, because under group normalization a rater flipping a 3 to a 4 inverts an
#    advantage sign
assert cfg['judge']['temperature'] == 0.0 and cfg['judge']['seed'] == 42

# -- the rubric must be the frozen one; load_train_template raises on drift, this reports it
text = load_train_template()
digest = hashlib.sha256(text.encode()).hexdigest()
frozen = json.loads(pathlib.Path('prompts/hashes.json').read_text())['templates']
assert digest == frozen['judge_train.txt']['sha256']
assert cfg['template_file'] == 'prompts/judge_train.txt', 'the eval rubric would be circular'

print(f"reward judge {cfg['judge']['model']}, distinct from {sorted(raters)}")
print(f"rubric verified {digest[:16]}")

In [ ]:
# -- the dev slice, against the manifest written when it was carved
man = json.loads(pathlib.Path('data/splits/rlsf_dev_manifest.json').read_text())
for name, want in man['hashes'].items():
    got = hashlib.sha256((pathlib.Path('data/splits') / name).read_bytes()).hexdigest()
    assert got == want, f'{name} differs from the manifest'
print(f"dev slice {man['counts']['rlsf_dev']} segments, {man['counts']['dev_works']} works")

# -- the slice is not unseen by the model; it selects weights, it does not measure them
print('\n'.join('  ' + c for c in man['caveats']))

p = plan(SEGMENTS, G)
print(f"\nplanned: {p['judge_calls']} judge calls, ~${p['est_usd']} "
      f"(pilot ceiling {cfg['rlsf']['pilot']['judge_calls']})")

---
## 3 — Free pass

In [ ]:
!python manage.py rlsf_smoke --config {CONFIG} --segments 4 --skip_judge \
    --out outputs/rlsf/smoke_free.jsonl

---
## 4 — Paid pass

In [ ]:
!python manage.py rlsf_smoke --config {CONFIG} --segments {SEGMENTS} --group_size {G} --yes

---
## 5 — Read the result


In [ ]:
log = json.loads(pathlib.Path('outputs/rlsf/smoke_steps.jsonl').read_text().splitlines()[0])
print(f"samples {log['n_samples']}  feasible {log['n_feasible']} "
      f"({log['n_feasible'] / log['n_samples']:.0%})")
print(f"reward mean {log['reward_mean']:+.3f}  sd {log['reward_sd']:.3f}")
print(f"length mean {log['length_mean']:.1f} words, ratio to reference "
      f"{log['length_ratio_mean']:.2f}")
print('\nraw component means:', {k: round(v, 3) for k, v in log['raw'].items()})
print('z-deviation from the register centroid:')
for k, v in log['z'].items():
    print(f"  {k:12s} {v:+.2f}")

In [ ]:
# The measured per-call rate. docs/budget.md carries an estimate over assumed token
# counts until this replaces it.
u = json.loads(pathlib.Path('outputs/rlsf/smoke_usage.json').read_text())
print(f"{u['calls']} calls, {u['prompt_tokens'] / u['calls']:.0f} in / "
      f"{u['completion_tokens'] / u['calls']:.0f} out per call")
print(f"measured ${u['per_call_usd']:.6f}/call against the $0.000114 assumed")
print(f"\nprojected at the 44,800-call ceiling: ${u['per_call_usd'] * 44_800:.2f} "
      f"(docs/budget.md records $5.11-$6.45)")

---
## 6 — What each failure means

**kiwi handshake** — the worker did not report ready inside 600 s. Usually a missing
`.venv-comet`, or the checkpoint download failing. Read the worker's stderr.

**reward variance** — more than a quarter of groups have no spread in their *combined*
reward across their *feasible* samples, so those samples carry no advantage and
contribute no gradient. Training would run, log plausible rewards, and learn nothing.

Read the per-component diagnostics printed above the verdicts to see which term went
flat, but do not read a flat component as a failure on its own: one component can
normalize to zeros while the others still spread the combined reward. That is also why
`--skip_judge` cannot fail this check by itself.

Two causes. Samples within a group are too similar — raise `rollout.temperature`, or
`group_size` up to the ceiling of 8. Or the length band is rejecting so much that groups
fall below the two feasible samples `group_normalize` needs. **Read the length band
verdict first**: it produces variance failures whose fix lies elsewhere.

**steplog written** — the run produced no record of itself. Check the `--out` path.

**length band** — half or more of the samples fall outside `[len_min_ratio,
len_max_ratio]`, so `on_violation: floor` dominates and the policy is learning the
thresholds rather than the weights. Look at `length_ratio_mean`: well above 1 usually
means the policy is padding or repeating, well below 1 means truncation.

The generation-length lever is `generator.max_tokens`, and it is 1024 in every condition
config. Changing it for RLSF alone would make the arm non-comparable on length against
the conditions it is measured against, and length is what the feasibility band gates on.
Move `len_min_ratio` / `len_max_ratio` instead, and record why.

Any change to `reward:` or `rollout:` is a design change. Record it in the DEVLOG with
what it invalidates, and re-run this smoke before training on it.